# AC SugarSims: Architecture Experiment
**18 runs** (6 conditions × 3 seeds × 3000 steps)

Tests the contribution of each architectural layer:
| # | Firms | Trust | HI | Objective | Tests |
|---|-------|-------|----|-----------|-------|
| C1 | Vanilla | None | No | SUM | True baseline |
| C2 | SEVC | None | No | SUM | +sustainable capitalism |
| C3 | SEVC | None | Yes | SUM | +horizon index |
| C4 | SEVC | Fuzzy 0.1 | No | SUM | +trust |
| C5 | SEVC | Fuzzy 0.1 | Yes | SUM | +trust +HI |
| C6 | SEVC | Fuzzy 0.1 | Yes | TOPO_X | +smart planner |


In [ ]:
# ── Hardware Detection ──
import subprocess, os

def detect_gpu():
    try:
        out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                       '--format=csv,noheader']).decode().strip()
        name, mem = out.split(',')
        mem_gb = int(mem.strip().split()[0]) / 1024
        print(f"GPU: {name.strip()} ({mem_gb:.0f} GB)")
        return name.strip()
    except:
        print("No GPU detected, using CPU")
        return None

gpu = detect_gpu()

# CPU info
cpu_count = os.cpu_count()
print(f"CPUs: {cpu_count}")

# Recommended parallelism (CPU-bound sim)
N_PARALLEL = min(6, max(1, cpu_count - 2))
print(f"Recommended parallel workers: {N_PARALLEL}")


In [ ]:
# ── Clone Repository ──
import os
if os.path.exists('ac_sugarsims'):
    !cd ac_sugarsims && git pull
else:
    !git clone https://github.com/caseymrobbins/ac_sugarsims.git

os.chdir('ac_sugarsims')
print(f"Working directory: {os.getcwd()}")
!ls *.py


In [ ]:
# ── Install Dependencies ──
!pip install -q mesa numpy pandas pyarrow


In [ ]:
# ── Verify All Modules Load ──
import importlib
modules = ['environment', 'agents', 'planner', 'metrics', 'trust',
           'innovation', 'sustainable_capitalism', 'horizon_index',
           'economy', 'banking']

ok = 0
for m in modules:
    try:
        importlib.import_module(m)
        print(f"  ✓ {m}")
        ok += 1
    except Exception as e:
        print(f"  ✗ {m}: {e}")

print(f"\n{ok}/{len(modules)} modules loaded")


In [ ]:
# ── Smoke Test (50 steps, vanilla) ──
from environment import EconomicModel

model = EconomicModel(seed=42, grid_width=40, grid_height=40,
                      n_workers=50, n_firms=5, n_landowners=3,
                      objective="SUM_RAW")

for i in range(50):
    model.step()

h = model.metrics_history[-1]
def _f(k):
    v = h.get(k, 'N/A')
    return f"{v:.3f}" if isinstance(v, float) else str(v)

print(f"Step 50 smoke test:")
print(f"  Workers: {_f('n_workers')}, Firms: {_f('n_firms')}")
print(f"  Unemployment: {_f('unemployment_rate')}")
print(f"  Worker min: {_f('worker_min')}, mean: {_f('worker_mean')}")
print(f"  Agency floor: {_f('agency_floor')}")
print(f"  HI: {_f('horizon_index')}")
print(f"  Trust planner: {_f('trust_planner')}")
print("\n✓ Smoke test passed")


## Run the Experiment
Upload `run_architecture_experiment.py` and `horizon_index.py` (corrected min-based HI) to the repo before running.

If files are already pushed to the repo, skip the upload cell below.

In [ ]:
# ── Upload corrected files (if not already in repo) ──
# Uncomment and run if you need to upload:

# from google.colab import files
# uploaded = files.upload()  # Upload horizon_index.py, run_architecture_experiment.py
# for name, data in uploaded.items():
#     with open(name, 'wb') as f:
#         f.write(data)
#     print(f"Saved: {name}")


In [ ]:
# ── Run the Full Experiment ──
# 18 runs × 3000 steps
# Estimated time: ~30-60 min on A100 with 4 workers

import time
t0 = time.time()

# Sequential is safer for Colab (multiprocessing can be tricky in notebooks)
# For parallel: !python run_architecture_experiment.py --parallel 4
!python run_architecture_experiment.py --parallel 1

elapsed = time.time() - t0
print(f"\nTotal wall time: {elapsed/60:.1f} minutes")


In [ ]:
# ── Load Results ──
import pandas as pd
import numpy as np
import os

raw_dir = "results/architecture/raw_data"
files = sorted([f for f in os.listdir(raw_dir) if f.endswith('.parquet')])
print(f"Found {len(files)} result files")

all_data = pd.concat([pd.read_parquet(f"{raw_dir}/{f}") for f in files], ignore_index=True)
print(f"Total rows: {len(all_data)}")
print(f"Conditions: {sorted(all_data['condition'].unique())}")
print(f"Seeds: {sorted(all_data['seed'].unique())}")


In [ ]:
# ── Staircase Visualization ──
import matplotlib.pyplot as plt

late = all_data[all_data['step'] >= 2500]

conditions_order = ['C1_vanilla_sum', 'C2_sevc_sum', 'C3_sevc_hi_sum',
                    'C4_sevc_trust_sum', 'C5_sevc_trust_hi_sum', 'C6_full_topo']
labels = ['C1: Vanilla', 'C2: +SEVC', 'C3: +HI', 'C4: +Trust', 'C5: +Trust+HI', 'C6: TOPO_X']

metrics = {
    'worker_min': ('Worker Floor', False),
    'worker_gini': ('Worker Gini', True),
    'unemployment_rate': ('Unemployment', True),
    'agency_floor': ('Agency Floor', False),
    'n_firms': ('Firm Count', False),
    'n_active_cartels': ('Active Cartels', True),
    'horizon_index': ('Horizon Index', False),
    'trust_institutional': ('Institutional Trust', False),
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Architecture Staircase: Each Layer\'s Contribution', fontsize=16, fontweight='bold')

for idx, (key, (title, lower_better)) in enumerate(metrics.items()):
    ax = axes[idx // 4][idx % 4]
    means = []
    stds = []
    for c in conditions_order:
        subset = late[late['condition'] == c]
        by_seed = subset.groupby('seed')[key].mean()
        means.append(by_seed.mean())
        stds.append(by_seed.std())

    colors = ['#ef4444' if lower_better else '#6366f1'] * 6
    colors[-1] = '#10b981'  # TOPO_X in green

    bars = ax.bar(range(len(conditions_order)), means, yerr=stds,
                  color=colors, alpha=0.8, capsize=3)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/architecture/staircase.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/architecture/staircase.png")


In [ ]:
# ── Trajectory Comparison ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Trajectory Over Time by Condition', fontsize=16, fontweight='bold')

traj_metrics = ['worker_min', 'worker_mean', 'unemployment_rate', 'all_gini',
                'agency_floor', 'horizon_index', 'n_firms', 'trust_institutional']

condition_colors = {
    'C1_vanilla_sum': '#94a3b8',
    'C2_sevc_sum': '#6366f1',
    'C3_sevc_hi_sum': '#8b5cf6',
    'C4_sevc_trust_sum': '#f59e0b',
    'C5_sevc_trust_hi_sum': '#f97316',
    'C6_full_topo': '#10b981',
}
condition_labels = {
    'C1_vanilla_sum': 'C1: Vanilla',
    'C2_sevc_sum': 'C2: +SEVC',
    'C3_sevc_hi_sum': 'C3: +HI',
    'C4_sevc_trust_sum': 'C4: +Trust',
    'C5_sevc_trust_hi_sum': 'C5: +T+HI',
    'C6_full_topo': 'C6: TOPO',
}

# Subsample for plotting
sample_step = 10
sampled = all_data[all_data['step'] % sample_step == 0]

for idx, key in enumerate(traj_metrics):
    ax = axes[idx // 4][idx % 4]
    for cond in conditions_order:
        subset = sampled[sampled['condition'] == cond]
        by_step = subset.groupby('step')[key].mean()
        ax.plot(by_step.index, by_step.values,
                color=condition_colors[cond],
                label=condition_labels[cond],
                linewidth=1.5, alpha=0.85)
    ax.set_title(key, fontsize=10, fontweight='bold')
    ax.grid(alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('results/architecture/trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/architecture/trajectories.png")


In [ ]:
# ── Download Results ──
import shutil
shutil.make_archive('architecture_results', 'zip', 'results/architecture')
print("Created: architecture_results.zip")

from google.colab import files
files.download('architecture_results.zip')
